> **Quick start — click *Run All* — no setup needed.**
> Cached PNG figures are displayed by default (`RERUN = False`).
> Set `RERUN = True` and re-run to regenerate figures from the source scripts.

In [ ]:
import subprocess, sys, pathlib
from IPython.display import Image, display

# locate repository root by searching upward for lunar/__init__.py
_here = pathlib.Path.cwd()
REPO = None
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Cannot find REPO root — run from inside Lunar-V2/")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS    = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts" / "phase2"


def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[warn] figure not found: {path}")
        return
    display(Image(str(path), width=900))
    if caption:
        from IPython.display import Markdown
        display(Markdown(f"*{caption}*"))


def run_script(name: str) -> None:
    """Run a Phase-2 script via subprocess."""
    script = SCRIPTS / name
    print(f"Running: {script}")
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=str(REPO),
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Script exited with code {result.returncode}: {script}")

print(f"REPO    = {REPO}")
print(f"FIGS    = {FIGS}  (exists={FIGS.exists()})")
print(f"SCRIPTS = {SCRIPTS}  (exists={SCRIPTS.exists()})")


# Phase 2 (PSR Shoemaker) — Figs 3, 4, 5

| Fig | Subject | Script | Runtime |
|-----|---------|--------|---------|
| 3 | Surface T(t) driven by ray-traced illumination | `psr_shoemaker/fig3_diurnal.py` | ~5–10 min |
| 4 | T(z) steady-state profile, Hayne vs M&S | `psr_shoemaker/figs45_subsurface.py` | <5 s |
| 5 | ΔT(z) = T_M&S − T_Hayne (1-D proxy for 2-D map) | `psr_shoemaker/figs45_subsurface.py` | <5 s |

Mirrors upstream `PSRShoemaker/UpdatedModel/heat1DShoemaker.m` using the same `shoemakerIllumination.mat` input.

In [ ]:
RERUN = False

## §1 — Upstream illumination input: `shoemakerIllumination.mat`

**Data provenance:** 697 samples spanning ~23 months at the Shoemaker crater PSR floor (lat = −87.91°, lon = 45.51°), precomputed by ray-tracing with LOLA topography. Zenodo DOI [10.5281/zenodo.12586656](https://doi.org/10.5281/zenodo.12586656).

| Field | Description |
|-------|-------------|
| `Q_visible` | Scattered visible flux from sunlit crater walls (W m⁻²) |
| `Q_ir` | Thermal IR re-emission from warm crater walls (W m⁻²) |
| `Q_total` | Sum; mean ≈ 0.141 W m⁻² |
| `t_jd` | Julian date of each sample |
| `T_reference` | `daveTemp` reference temperature; mean ≈ 34.2 K |

Loaded via `lunar.illumination.load_shoemaker_illumination()`; local path: `data/upstream/martinez2021/shoemakerIllumination.mat`.

In [ ]:
from lunar.illumination import load_shoemaker_illumination
import numpy as np
import matplotlib.pyplot as plt

sh = load_shoemaker_illumination()
days = np.arange(len(sh["t_jd"]))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, constrained_layout=True)

ax1.fill_between(days, sh["Q_ir"], alpha=0.5, color="#d62728", label="IR re-emission from walls")
ax1.fill_between(days, sh["Q_visible"], alpha=0.5, color="#1f77b4", label="Scattered visible")
ax1.plot(days, sh["Q_total"], "k-", lw=1.2,
         label=f"Q_total (mean {sh['Q_total'].mean():.3f} W/m²)")
ax1.set_ylabel("Flux (W/m²)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_title("Shoemaker illumination (real ray-traced upstream data)")

ax2.plot(days, sh["T_reference"], color="0.3", lw=1.2, marker=".", ms=3,
         label=f"daveTemp (mean {sh['T_reference'].mean():.1f} K)")
ax2.set_xlabel("Sample index (697 over ~23 months)")
ax2.set_ylabel("T reference (K)")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.show()

print(f"Q_total: mean={sh['Q_total'].mean():.4f}, peak={sh['Q_total'].max():.4f} W/m²")
print(f"daveTemp: mean={sh['T_reference'].mean():.2f}, "
      f"range=[{sh['T_reference'].min():.1f}, {sh['T_reference'].max():.1f}] K")

## §2 — Figure 3 · Shoemaker surface T(t), Hayne vs M&S

The Phase-1 1-D solver is driven by `Q_total(t)` resampled to 10-minute steps. Configuration:

- `spinup_depth_m = 0.10 m` (top-layer convergence criterion)
- Convergence tolerance: `tol = 0.05 K`
- Both Hayne 2017 and M&S 2021 K-models run back-to-back

Simulated surface temperature compared to `daveTemp`. This is a **faithful replication** — the same `Q(t)` time series that the upstream `heat1DShoemaker.m` consumed.

In [ ]:
if RERUN:
    print("Fig 3 takes ~5-10 min "
          "(time-domain spinup x2 K models, 697-day span at 10-min steps).")
    run_script("psr_shoemaker/fig3_diurnal.py")
show_figure("phase2_fig3_shoemaker_diurnal.png",
            "Fig 3 — Shoemaker surface T(t) driven by real upstream illumination")

## §3 — Figures 4 & 5 · Shoemaker T(z) and ΔT(z)

Steady-state 1-D heat equation on the geometric depth grid:

$$\frac{d}{dz}\!\left[K(T,z)\frac{dT}{dz}\right] = 0$$

Boundary conditions:

- **Surface (Dirichlet):** T = `daveTemp` mean = 34.2 K
- **Bottom:** geothermal flux Q_geo = 18 mW m⁻²

**Key diagnostic:** `ΔT(4 m)` = T_M&S − T_Hayne at 4 m depth — printed to stdout when `figs45_subsurface.py` is executed. This scalar is the 1-D proxy for the 2-D subsurface temperature contrast mapped in the full Phase-3 polar run.

In [ ]:
if RERUN:
    run_script("psr_shoemaker/figs45_subsurface.py")
show_figure("phase2_fig4_shoemaker_Tz.png", "Fig 4 — Shoemaker T(z), Hayne vs M&S")
show_figure("phase2_fig5_dT_vs_depth.png", "Fig 5 — ΔT vs depth, 1-D proxy for 2-D 4-m map")

## §4 — References

- **Martinez & Siegler 2021** — *J. Geophys. Res. Planets*, 126, e2020JE006792. [doi:10.1029/2020JE006792](https://doi.org/10.1029/2020JE006792)
- **Upstream data (shoemakerIllumination.mat)** — Zenodo [10.5281/zenodo.12586656](https://doi.org/10.5281/zenodo.12586656)
- **Hayne et al. 2017** — *J. Geophys. Res. Planets*, 122, 2371–2400. [doi:10.1002/2017JE005387](https://doi.org/10.1002/2017JE005387)
- **Vasavada et al. 2012** — *J. Geophys. Res. Planets*, 117, E00H18. [doi:10.1029/2011JE003987](https://doi.org/10.1029/2011JE003987)
- **Mazarico et al. 2011** — *Icarus*, 211, 1066–1081. [doi:10.1016/j.icarus.2010.10.030](https://doi.org/10.1016/j.icarus.2010.10.030)
- **Diviner Lunar Radiometer Experiment** — GCP PDS archive, [pds-geosciences.wustl.edu](https://pds-geosciences.wustl.edu/)